<a href="https://colab.research.google.com/github/NhanChanDeo/VQA_Final_Thesis/blob/main/DATASETS%20SCHEMA%20NORMALIZATION/Step7_Sharding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 7: In-Place Parquet Resharding & Hub Synchronization (CrossVQA)

This notebook resolves the **`TooBigContentError` (HTTP 501 / Scan size limit exceeded)** on the Hugging Face Dataset Viewer and PyTorch DataLoader pipelines for:
- **TextVQA Canonical:** `gamusa/textvqa-canonical`
- **DocVQA Canonical:** `gamusa/docvqa-canonical`

### OOM-Prevention & Streaming Architecture
1. **Constant Memory Streaming:** Reads monolithic Parquet files with `buffer_size=8MB`, `pre_buffer=False`, and `use_threads=False` to prevent multi-gigabyte column buffer blowups in RAM.
2. **Incremental ParquetWriter:** Streams directly row-group by row-group (`batch_size=row_group_size`), keeping RAM footprint under ~200 MB throughout the entire run.
3. **Automatic Memory Pool Reclamation:** Calls `pa.default_memory_pool().release_unused()` and `gc.collect()` between splits.
4. **Disk Quota Protection:** Automatically removes temporary downloaded monolithic files after each split is resharded to prevent Colab disk exhaustion.

In [ ]:
# 1. Environment Setup & Authentication
!pip install -q --upgrade huggingface_hub pyarrow pandas tqdm

import os
import gc
import math
from pathlib import Path
from typing import List, Optional
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm.auto import tqdm
from huggingface_hub import HfApi, login, hf_hub_download

# Dual-environment Token Auth (Colab userdata + local env fallback)
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.getenv('HF_TOKEN')

if not HF_TOKEN:
    raise ValueError("Please configure HF_TOKEN in Colab Secrets or environment variables!")

login(token=HF_TOKEN)
api = HfApi(token=HF_TOKEN)
username = api.whoami()['name']
print(f"✓ Authenticated as: {username}")

# Workspace directories
WORK_DIR = Path("./crossvqa_reshard_workspace")
RAW_DIR = WORK_DIR / "monolithic_inputs"
SHARD_DIR = WORK_DIR / "shards"
RAW_DIR.mkdir(parents=True, exist_ok=True)
SHARD_DIR.mkdir(parents=True, exist_ok=True)
print(f"✓ Workspace configured at: {WORK_DIR.resolve()}")

In [ ]:
# 2. Constant-Memory Streaming Resharding Engine (OOM-Safe)
def reshard_parquet_file(
    input_path: Path,
    output_dir: Path,
    split_name: str,
    shard_size: int = 2000,
    row_group_size: int = 250,
    compression: str = 'zstd',
    delete_source: bool = False
) -> List[Path]:
    """
    Streams a monolithic Parquet file with constant RAM footprint (< 200MB),
    even if the source file contains 20GB+ in a single row group.
    """
    # buffer_size > 0 streams pages incrementally; pre_buffer=False stops loading whole column chunks
    pfile = pq.ParquetFile(input_path, buffer_size=8 * 1024 * 1024, pre_buffer=False)
    total_rows = pfile.metadata.num_rows
    total_shards = max(1, math.ceil(total_rows / shard_size))
    schema = pfile.schema_arrow
    
    print(f"\n{'='*25} Resharding: {input_path.name} {'='*25}")
    print(f"Rows: {total_rows} | Target Shards: {total_shards} | Source Row Groups: {pfile.num_row_groups}")
    print(f"Config: shard_size={shard_size}, row_group_size={row_group_size}, compression={compression}")
    
    created_shards = []
    writer = None
    current_shard_idx = -1
    rows_in_current_shard = shard_size  # Triggers immediate creation of first shard
    
    def _close_shard(w, idx):
        w.close()
        p = created_shards[-1]
        size_mb = p.stat().st_size / (1024 ** 2)
        print(f"  ✓ Shard {idx + 1}/{total_shards}: {p.name} ({size_mb:.1f} MB)")
        
    total_batches = math.ceil(total_rows / row_group_size)
    
    # use_threads=False guarantees sequential decoding without multiplying memory buffers
    for batch in tqdm(
        pfile.iter_batches(batch_size=row_group_size, use_threads=False),
        total=total_batches,
        desc=f"Sharding {split_name}"
    ):
        if rows_in_current_shard >= shard_size:
            if writer is not None:
                _close_shard(writer, current_shard_idx)
            current_shard_idx += 1
            shard_name = f"{split_name}-{current_shard_idx:05d}-of-{total_shards:05d}.parquet"
            shard_path = output_dir / shard_name
            writer = pq.ParquetWriter(
                shard_path,
                schema,
                compression=compression,
                write_page_index=True  # Enables DuckDB page-level random access for HF Dataset Viewer
            )
            created_shards.append(shard_path)
            rows_in_current_shard = 0
            
        writer.write_batch(batch, row_group_size=row_group_size)
        rows_in_current_shard += batch.num_rows
        del batch
        
    if writer is not None:
        _close_shard(writer, current_shard_idx)
        
    # Explicit memory cleanup between files
    del pfile
    gc.collect()
    pa.default_memory_pool().release_unused()
    
    # Disk protection: remove temporary monolithic file downloaded from Hub
    if delete_source:
        try:
            input_path.unlink()
            print(f"  🗑️ Deleted downloaded source file to save disk: {input_path.name}")
        except Exception as e:
            print(f"  Warning: Could not delete {input_path.name}: {e}")
            
    return created_shards

In [ ]:
# 3. Pipeline for TextVQA Canonical (gamusa/textvqa-canonical)
TEXTVQA_REPO = f"{username}/textvqa-canonical"
textvqa_shard_dir = SHARD_DIR / "textvqa"
textvqa_shard_dir.mkdir(parents=True, exist_ok=True)

# TextVQA splits: train (34,602 rows), validation (5,000 rows), test (5,734 rows)
# Recommended shard size: 2500 samples per shard (~300-400 MB compressed)
textvqa_splits = [
    {"split": "train", "src_file": "data/train-00000.parquet", "shard_size": 2500},
    {"split": "validation", "src_file": "data/validation-00000.parquet", "shard_size": 2500},
    {"split": "test", "src_file": "data/test-00000.parquet", "shard_size": 2500}
]

print(f"=== Processing {TEXTVQA_REPO} ===")
textvqa_generated_shards = {}

for cfg in textvqa_splits:
    split = cfg["split"]
    local_candidate = Path(f"./crossvqa_workspace/normalized/textvqa_{split}.parquet")
    is_downloaded = False
    
    if local_candidate.exists():
        input_p = local_candidate
        print(f"Found local normalized file for TextVQA [{split}]: {input_p}")
    else:
        print(f"Downloading {cfg['src_file']} from {TEXTVQA_REPO}...")
        input_p = Path(hf_hub_download(
            repo_id=TEXTVQA_REPO,
            repo_type="dataset",
            filename=cfg["src_file"],
            local_dir=str(RAW_DIR / "textvqa")
        ))
        is_downloaded = True
    
    shards = reshard_parquet_file(
        input_path=input_p,
        output_dir=textvqa_shard_dir,
        split_name=split,
        shard_size=cfg["shard_size"],
        row_group_size=250,
        compression="zstd",
        delete_source=is_downloaded  # Reclaim disk space if downloaded from Hub
    )
    textvqa_generated_shards[split] = shards
    
    gc.collect()
    pa.default_memory_pool().release_unused()

print("✓ TextVQA resharding complete!")

In [ ]:
# 4. Pipeline for DocVQA Canonical (gamusa/docvqa-canonical)
DOCVQA_REPO = f"{username}/docvqa-canonical"
docvqa_shard_dir = SHARD_DIR / "docvqa"
docvqa_shard_dir.mkdir(parents=True, exist_ok=True)

# DocVQA splits: train (39,463 rows), validation (5,349 rows), test (5,188 rows)
# DocVQA images have higher resolution; 1500 samples per shard keeps files ~400-500 MB
docvqa_splits = [
    {"split": "train", "src_file": "data/train-00000.parquet", "shard_size": 1500},
    {"split": "validation", "src_file": "data/validation-00000.parquet", "shard_size": 1500},
    {"split": "test", "src_file": "data/test-00000.parquet", "shard_size": 1500}
]

print(f"=== Processing {DOCVQA_REPO} ===")
docvqa_generated_shards = {}

for cfg in docvqa_splits:
    split = cfg["split"]
    local_candidate = Path(f"./crossvqa_workspace/normalized/docvqa_{split}.parquet")
    is_downloaded = False
    
    if local_candidate.exists():
        input_p = local_candidate
        print(f"Found local normalized file for DocVQA [{split}]: {input_p}")
    else:
        print(f"Downloading {cfg['src_file']} from {DOCVQA_REPO}...")
        input_p = Path(hf_hub_download(
            repo_id=DOCVQA_REPO,
            repo_type="dataset",
            filename=cfg["src_file"],
            local_dir=str(RAW_DIR / "docvqa")
        ))
        is_downloaded = True
    
    shards = reshard_parquet_file(
        input_path=input_p,
        output_dir=docvqa_shard_dir,
        split_name=split,
        shard_size=cfg["shard_size"],
        row_group_size=200,
        compression="zstd",
        delete_source=is_downloaded  # Critical for DocVQA train (frees ~19.6 GB disk space!)
    )
    docvqa_generated_shards[split] = shards
    
    gc.collect()
    pa.default_memory_pool().release_unused()

print("✓ DocVQA resharding complete!")

In [ ]:
# 5. Verification: Total Row Count Conservation
def verify_shards(shard_list: List[Path], expected_rows: int, dataset_name: str):
    actual_rows = 0
    for sp in shard_list:
        pf = pq.ParquetFile(sp)
        actual_rows += pf.metadata.num_rows
    assert actual_rows == expected_rows, f"Mismatch for {dataset_name}: expected {expected_rows}, got {actual_rows}"
    print(f"✓ {dataset_name}: 100% Match ({actual_rows}/{expected_rows} rows across {len(shard_list)} shards)")

print("=== VERIFYING TEXTVQA SHARDS ===")
verify_shards(textvqa_generated_shards["train"], 34602, "TextVQA Train")
verify_shards(textvqa_generated_shards["validation"], 5000, "TextVQA Validation")
verify_shards(textvqa_generated_shards["test"], 5734, "TextVQA Test")

print("\n=== VERIFYING DOCVQA SHARDS ===")
verify_shards(docvqa_generated_shards["train"], 39463, "DocVQA Train")
verify_shards(docvqa_generated_shards["validation"], 5349, "DocVQA Validation")
verify_shards(docvqa_generated_shards["test"], 5188, "DocVQA Test")

In [ ]:
# 6. Synchronize Shards to Hugging Face Hub & Delete Obsolete Monolithic Files
def sync_sharded_repo_to_hub(repo_name: str, shard_dir: Path, task_category: str, tags: str):
    full_repo = f"{username}/{repo_name}"
    print(f"\n{'='*25} Syncing {full_repo} {'='*25}")
    
    # 1. Update README.md with standard glob data_files
    readme_content = f"""---
configs:
- config_name: default
  data_files:
  - split: train
    path: data/train-*
  - split: validation
    path: data/validation-*
  - split: test
    path: data/test-*
task_categories:
- {task_category}
tags:
{tags}
---
# {repo_name}
Normalized Canonical Dataset for CrossVQA Benchmark with Page-Indexed Parquet Shards.\n"""
    readme_path = WORK_DIR / f"README_{repo_name}.md"
    readme_path.write_text(readme_content, encoding='utf-8')
    api.upload_file(
        path_or_fileobj=str(readme_path),
        path_in_repo="README.md",
        repo_id=full_repo,
        repo_type="dataset"
    )
    print("  ✓ Updated README.md configuration metadata.")
    
    # 2. Upload all newly partitioned shards
    all_shards = sorted(shard_dir.glob("*.parquet"))
    for shard in tqdm(all_shards, desc=f"Uploading shards for {repo_name}"):
        dest_path = f"data/{shard.name}"
        api.upload_file(
            path_or_fileobj=str(shard),
            path_in_repo=dest_path,
            repo_id=full_repo,
            repo_type="dataset"
        )
    print(f"  ✓ Uploaded {len(all_shards)} shards to 'data/' folder.")
    
    # 3. Clean up old monolithic files to avoid glob collisions
    repo_files = api.list_repo_files(repo_id=full_repo, repo_type="dataset")
    obsolete_files = [
        "data/train-00000.parquet",
        "data/validation-00000.parquet",
        "data/test-00000.parquet"
    ]
    for old_f in obsolete_files:
        if old_f in repo_files:
            try:
                api.delete_file(path_in_repo=old_f, repo_id=full_repo, repo_type="dataset")
                print(f"  🗑️ Deleted obsolete monolithic file: {old_f}")
            except Exception as e:
                print(f"  Warning: Could not delete {old_f}: {e}")
                
    print(f"✓ Dataset is fully live and compliant: https://huggingface.co/datasets/{full_repo}")

# Sync TextVQA Canonical
sync_sharded_repo_to_hub(
    repo_name="textvqa-canonical",
    shard_dir=textvqa_shard_dir,
    task_category="visual-question-answering",
    tags="- textvqa\n- scene-text\n- canonical\n- crossvqa"
)

# Sync DocVQA Canonical
sync_sharded_repo_to_hub(
    repo_name="docvqa-canonical",
    shard_dir=docvqa_shard_dir,
    task_category="visual-question-answering",
    tags="- docvqa\n- document-ai\n- canonical\n- crossvqa"
)

In [ ]:
# 7. Live Dataset Viewer Verification via HF API
import urllib.request
import json

print("=== TESTING HUGGING FACE DATASET VIEWER FIRST-ROWS API ===")
for repo in [f"{username}/textvqa-canonical", f"{username}/docvqa-canonical"]:
    url = f"https://datasets-server.huggingface.co/first-rows?dataset={repo}&config=default&split=train"
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req) as resp:
            res = json.loads(resp.read().decode('utf-8'))
            rows = res.get('rows', [])
            print(f"✓ {repo}: HTTP {resp.status} - First {len(rows)} rows loaded successfully! (TooBigContentError RESOLVED)")
    except urllib.error.HTTPError as e:
        body = e.read().decode('utf-8', errors='replace')
        print(f"❌ {repo}: HTTP {e.code} - {body[:300]}")
    except Exception as e:
        print(f"❌ {repo}: Error - {e}")